In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
"""
.. _uoi_lasso:

UoI-Lasso for sparse, minimal bias, regression
=============================r[i================

This example with demonstrate the ability of UoI-Lasso to recover sparse
models with minimal bias.

"""

###############################################################################
# Load synthetic data
# -------------------
#
# The synthetic data will have 40 features, 10 of which are informative and
# 1 response variable.


import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from sklearn.linear_model import LinearRegression, LassoCV

from pyuoi.linear_model.lasso_VAR import UoI_Lasso_VAR
from pyuoi.datasets import make_linear_regression
import pandas as pd
from scipy.linalg import solve_discrete_lyapunov
from var_utils import *
from tqdm import tqdm

In [3]:
# for GLM
from pyuoi.linear_model.poisson_VAR import (Poisson,
                                UoI_Poisson_VAR)
from pyuoi.datasets import make_poisson_regression
from numpy.testing import (assert_allclose,
                           assert_equal,
                           assert_raises)

### Fitting

In [14]:
n_features = 10
n_samples = 200
lag = 1


data, transition_matrices, cov = generate_sparse_stationary_var_process(
    n_features,
    n_samples,
    lag=lag,
    sparsity=0.5,
    spectral_radius=0.98,  # Ensures stationarity
    process_type='gaussian'

)

0.9799999999999983


In [ ]:
dense_matrices = [M.toarray() for M in  transition_matrices]
# vecortized the ground truth transition matrices
B_truth = np.vstack([m.T for m in dense_matrices]).T.flatten()     
X,Y = vectorization(data, lag)

TypeError: generate_sparse_stationary_var_process() got an unexpected keyword argument 'random_seed'

In [5]:
uoi_lasso = UoI_Lasso_VAR(n_features, n_boots_sel=10, n_boots_est=50, stability_selection = 1)
uoi_lasso.fit(X, Y)
B_model = uoi_lasso.coef_

In [6]:
uoi_lasso.intercept_

array([0.0725362])

In [7]:
assert_allclose(B_truth, B_model, atol=0.5)

In [8]:
selection_accuracy(B_truth, B_model)

0.5714285714285714

In [9]:
# estimation error
est_mask = B_model != 0
np.linalg.norm(B_truth*est_mask - B_model)**2

0.24131049024297846

### Performance check fitting

In [4]:
# n_features = 10
# n_samples = 500
lag = 1

for n_features in [5,10,20,40,50]:
    for n_samples in [50,100,200]:

        data, transition_matrices, cov = generate_sparse_stationary_var_process(
            n_features,
            n_samples,
            lag=lag,
            sparsity=0.5,
            spectral_radius=0.9,  # Ensures stationarity
            process_type='gaussian'

        )
        
        dense_matrices = [M.toarray() for M in  transition_matrices]
        # vecortized the ground truth transition matrices
        B_truth = np.vstack([m.T for m in dense_matrices]).T.flatten()     
        X,Y = vectorization(data, lag)
        
        B_model = []
        
        for _ in tqdm(range(50)):
            uoi_lasso = UoI_Lasso_VAR(n_real_features = n_features, fit_VAR = True, n_boots_sel=10, n_boots_est=10, stability_selection = 0.7)
            uoi_lasso.fit(X, Y)
            B_model.append(uoi_lasso.coef_)
            
            #sa.append(selection_accuracy(B_truth, B_model))
    
        np.save("data/B_truth_"+str(n_features)+"_"+str(n_samples), B_truth)
        np.save("data/B_model_"+str(n_features)+"_"+str(n_samples), B_model)

  0%|                                                                                                       | 0/50 [00:00<?, ?it/s]/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.461e-01, tolerance: 1.746e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.954e-01, tolerance: 1.746e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objec

/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.939e-01, tolerance: 1.779e-01
  model = cd_fast.enet_coordinate_descent(
  4%|███▌                                                                                      | 2/50 [45:22<18:08:12, 1360.25s/it]/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.446e-01, tolerance: 1.791e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objec

/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.347e-01, tolerance: 1.711e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.209e-01, tolerance: 1.787e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing 

/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.913e-01, tolerance: 1.798e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.543e-01, tolerance: 1.798e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing 

/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.681e-01, tolerance: 1.721e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.051e-01, tolerance: 1.815e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing 

...

/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.041e-01, tolerance: 1.732e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.568e-01, tolerance: 1.775e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing 

/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.742e-01, tolerance: 1.771e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.867e-01, tolerance: 1.771e-01
  model = cd_fast.enet_coordinate_descent(
 92%|████████████████████████████████████████████████████████████████████████████████       | 46/50 [16:37:32<1:43:10, 1547.59s/it]/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objec

/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.083e-01, tolerance: 1.777e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.978e-01, tolerance: 1.777e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing 

/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.742e-01, tolerance: 1.779e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.320e-01, tolerance: 1.779e-01
  model = cd_fast.enet_coordinate_descent(
/Users/yao/anaconda3/envs/py39/lib/python3.9/site-packages/sklearn/linear_model/_coordinate_descent.py:697: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing 

LinAlgError: SVD did not converge in Linear Least Squares

In [245]:
evaluate_sparse_var_estimation(B_truth, B_model)

{'precision': 0.5116279069767442,
 'recall': 0.7586206896551724,
 'f1': 0.6111111111111112,
 'support_error': 0.28,
 'mae_nonzero': 0.05693006178445801,
 'rmse_nonzero': 0.07365923936695601,
 'mape_nonzero': 46.76520767257737,
 'true_sparsity': 0.71,
 'estimated_sparsity': 0.5700000000000001}